# SHine-K — URFD **FULL 70-sequence** fall-detection evaluation (Colab)

**사용법: 런타임 → 모두 실행(Run all) 한 번이면 끝납니다.** (CPU 런타임 약 1–2시간, T4 GPU 약 20–40분)

- 낙상 30 + 일상동작(ADL) 40 = **전체 70시퀀스**를, 논문과 동일한 배포 임계값(sens=1.0, 재튜닝 없음)으로 평가합니다.
- 평가 셀은 `SHine-K_reproducibility_colab.ipynb`의 원본 셀 **그대로**이며, 이 노트북은 환경변수만 30/40으로 설정합니다.
- 결과물: `eval_metrics.json` · `eval_per_sequence.csv` · `eval_confusion_matrix.png` · `run_manifest.json`
  — 실행이 끝나면 run 폴더 zip이 자동 다운로드되고, Drive가 연결되면 `코드/output/`에도 미러됩니다.
- **정직성 원칙**: 결과가 부분집합(8+8) 결과보다 나빠져도 그대로 보고합니다. 그것이 투고 전에 이 평가를 돌리는 이유입니다.


In [9]:
# === FULL-RUN CONFIG — the only change vs. the original notebook ===
import os
os.environ["URFD_N_FALL"] = "30"   # full dataset: 30 fall sequences
os.environ["URFD_N_ADL"]  = "40"   # full dataset: 40 ADL sequences
os.environ["URFD_STRIDE"] = "1"    # every frame, as in the paper's run
print("URFD full-run config:", {k: os.environ[k] for k in ("URFD_N_FALL","URFD_N_ADL","URFD_STRIDE")})

URFD full-run config: {'URFD_N_FALL': '30', 'URFD_N_ADL': '40', 'URFD_STRIDE': '1'}


In [10]:
# === RUN SETUP — per-run output folder + provenance (added) ===
import os, sys, json, time, platform, subprocess, hashlib, shutil, random
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np

SEED = 12345
random.seed(SEED); np.random.seed(SEED)   # figures/GIFs are deterministic; timing varies (reported as variance)

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False
IN_COLAB = _in_colab()

# Where the repo lives (for native Mac runs and for the Drive mirror)
REPO_LOCAL = Path("/Users/y3korea/Library/CloudStorage/GoogleDrive-y3korea@gmail.com/내 드라이브/완석_구글자료/연구자료/20260721_컴퓨터정보_논문3/코드")
DRIVE_REPO = None
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")               # idempotent; no force_remount
        _cands = [Path("/content/drive/MyDrive/완석_구글자료/연구자료/20260721_컴퓨터정보_논문3/코드"),
                  Path("/content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha/4. 제주학술대회/컴퓨터정보_논문3/코드")]
        DRIVE_REPO = next((c for c in _cands if c.exists()), None)
    except Exception as e:
        print("Drive mount skipped:", e)

# Primary write target: fast + never truncates. Colab=/content; local Mac=repo output/.
if IN_COLAB:
    BASE_OUTPUT = Path("/content/output")
elif REPO_LOCAL.exists():
    BASE_OUTPUT = REPO_LOCAL / "output"
else:
    BASE_OUTPUT = Path("./output")
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Timestamp: folder name in KST (author-friendly); manifest records BOTH utc & kst (Colab clock is UTC).
KST = timezone(timedelta(hours=9))
_utc = datetime.now(timezone.utc)
_kst = _utc.astimezone(KST)
RUN_ID = "run_" + _kst.strftime("%Y%m%d_%H%M%S")
RUN_DIR = BASE_OUTPUT / RUN_ID
FIG_DIR = RUN_DIR / "figures"
REC_DIR = FIG_DIR / "recovery"
LOG_DIR = RUN_DIR / "logs"
for d in (FIG_DIR, REC_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# The figure/GIF cells read these env vars (default ./figures if run standalone)
os.environ["OUT_FIG"] = str(FIG_DIR)
os.environ["OUT_REC"] = str(REC_DIR)

RUN_TIMES = {"utc_iso": _utc.isoformat(), "kst_iso": _kst.isoformat(), "tz_offset": "+09:00", "epoch": _utc.timestamp()}
print("RUN_ID   :", RUN_ID)
print("RUN_DIR  :", RUN_DIR)
print("IN_COLAB :", IN_COLAB, "| Drive mirror:", "yes" if DRIVE_REPO else "no")


RUN_ID   : run_20260723_135028
RUN_DIR  : /content/output/run_20260723_135028
IN_COLAB : True | Drive mirror: no


In [11]:
# Eval-only deps (kept out of the top install so figures/bench stay light). First Colab run: ~1-2 min.
!pip -q install tensorflow tensorflow-hub opencv-python-headless numpy matplotlib 2>/dev/null
print("eval deps ready")


eval deps ready


In [12]:
# === URFD fall-detection evaluation (guarded: skips gracefully if TF / network / data unavailable) ===
EVAL = None
try:
    import io, csv, json, math, zipfile, urllib.request
    import numpy as np
    import tensorflow as tf, tensorflow_hub as hub
    import cv2

    N_FALL = int(os.environ.get("URFD_N_FALL", "8"))   # full dataset = 30
    N_ADL  = int(os.environ.get("URFD_N_ADL",  "8"))   # full dataset = 40
    FRAME_STRIDE = int(os.environ.get("URFD_STRIDE", "1"))
    BASE_URL = "https://fenix.ur.edu.pl/~mkepski/ds/data"  # https: server 301-redirects http
    CACHE = BASE_OUTPUT / "_urfd_cache"; CACHE.mkdir(parents=True, exist_ok=True)

    print("Loading MoveNet MultiPose Lightning (matches the deployed model) ...")
    _net = hub.load("https://tfhub.dev/google/movenet/multipose/lightning/1")
    _infer = _net.signatures["serving_default"]

    def movenet_kps(frame_rgb):
        # -> (kps[17x(x_px,y_px,score)], W, H) for the top-scoring person, or None
        H, W = frame_rgb.shape[:2]
        img = tf.image.resize_with_pad(tf.expand_dims(frame_rgb, 0), 256, 256)
        img = tf.cast(img, dtype=tf.int32)
        res = _infer(tf.constant(img))["output_0"].numpy()[0]   # (6,56)
        best, bs = None, -1.0
        for person in res:
            kp = person[:51].reshape(17, 3)                      # [y,x,score] in padded 256 space
            sc = float(kp[:, 2].mean())
            if sc > bs: bs, best = sc, kp
        if best is None: return None
        scale = 256.0 / max(H, W); padx = (256 - W*scale)/2; pady = (256 - H*scale)/2
        out = np.zeros((17, 3))
        for i in range(17):
            yn, xn, s = best[i]
            out[i] = [(xn*256 - padx)/scale, (yn*256 - pady)/scale, s]
        return out, W, H

    # ---- faithful port of analyzePerson() from worksite_multi.html (deployed defaults, sens=1.0) ----
    ACT_IDX = [0, 5, 6, 9, 10, 11, 12, 15, 16]
    class FallSM:
        def __init__(self, fps=30.0, sens=1.0):
            self.fps=float(fps); self.sens=float(sens); self.reset()
        def reset(self):
            self.t=0.0; self.frame=-1; self.downSince=None; self.staticSince=None
            self.hipHist=[]; self.lastKp=None; self.state="normal"
            self.triggered=False; self.trigger_frame=None
        def _v(self, kps, i):
            x,y,s = kps[i]; return (x,y) if s>0.3 else None
        def step(self, kps, W, H):
            self.frame += 1; self.t += 1.0/self.fps; now = self.t*1000.0
            shL=self._v(kps,5); shR=self._v(kps,6); hipL=self._v(kps,11); hipR=self._v(kps,12)
            if shL is None or shR is None:
                self.state="normal"; self.lastKp=kps; return self.state
            shX=(shL[0]+shR[0])/2; shY=(shL[1]+shR[1])/2
            if hipL and hipR: hpX=(hipL[0]+hipR[0])/2; hpY=(hipL[1]+hipR[1])/2
            else:
                sw=math.hypot(shR[0]-shL[0], shR[1]-shL[1]); hpX=shX; hpY=shY+sw*1.4
            dx=shX-hpX; dy=shY-hpY
            tilt=math.degrees(math.atan2(abs(dx), abs(dy)))
            xs=[]; ys=[]
            for j in range(17):
                if kps[j][2]>0.3: xs.append(kps[j][0]); ys.append(kps[j][1])
            aspect=((max(xs)-min(xs))/max(1.0,(max(ys)-min(ys)))) if len(xs)>=2 else 0.0
            self.hipHist.append((hpY/H, self.t))
            if len(self.hipHist)>8: self.hipHist.pop(0)
            vel=0.0
            if len(self.hipHist)>=2:
                a=self.hipHist[0]; z=self.hipHist[-1]; vel=(z[0]-a[0])/max(0.001,(z[1]-a[1]))
            mv=0.0
            if self.lastKp is not None:
                sm=0.0; nm=0
                for i in ACT_IDX:
                    c=kps[i]; l=self.lastKp[i]
                    if c[2]>0.3 and l[2]>0.3:
                        sm+=math.hypot((c[0]-l[0])/W, (c[1]-l[1])/H); nm+=1
                mv=(sm/nm) if nm else 0.0
            self.lastKp=kps
            TILT_DOWN=52*self.sens; ASP_DOWN=1.0*self.sens; VEL_DROP=0.9/self.sens
            isDown=(tilt>TILT_DOWN) or (aspect>ASP_DOWN); rapid=vel>VEL_DROP
            if isDown:
                if not self.downSince: self.downSince=now
                self.state="fall" if (now-self.downSince>700) else "warn"
            elif rapid:
                self.state="warn"; self.downSince=None
            else:
                self.state="normal"; self.downSince=None
            if self.state=="normal":
                if mv<0.0045:
                    if not self.staticSince: self.staticSince=now
                    ss=(now-self.staticSince)/1000.0
                    if mv<0.0016 and ss>12: self.state="inactive"
                    elif ss>25: self.state="sedentary"
                else: self.staticSince=None
            else: self.staticSince=None
            if self.state in ("fall","inactive") and not self.triggered:
                self.triggered=True; self.trigger_frame=self.frame
            return self.state

    def load_frames(name):
        url=f"{BASE_URL}/{name}-cam0-rgb.zip"; zp=CACHE/f"{name}.zip"
        if not zp.exists(): urllib.request.urlretrieve(url, zp)
        frames=[]
        with zipfile.ZipFile(zp) as z:
            ns=sorted(n for n in z.namelist() if n.lower().endswith((".png",".jpg",".jpeg")))
            for k,n in enumerate(ns):
                if k % FRAME_STRIDE: continue
                arr=cv2.imdecode(np.frombuffer(z.read(n), np.uint8), cv2.IMREAD_COLOR)
                if arr is not None: frames.append(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))
        return frames

    seqs=[(f"fall-{i:02d}",1) for i in range(1,N_FALL+1)] + [(f"adl-{i:02d}",0) for i in range(1,N_ADL+1)]
    rows=[]; TP=FP=FN=TN=0; lat=[]
    for name,gt in seqs:
        try:
            frames=load_frames(name)
        except Exception as e:
            print("  skip", name, "-", e); continue
        if not frames: print("  no frames", name); continue
        sm=FallSM(fps=30.0/FRAME_STRIDE)
        for f in frames:
            r=movenet_kps(f)
            if r is None: sm.step([(0,0,0.0)]*17, f.shape[1], f.shape[0])
            else: sm.step(r[0], r[1], r[2])
        pred=1 if sm.triggered else 0
        rows.append({"seq":name,"gt":gt,"pred":pred,"trigger_frame":sm.trigger_frame,"frames":len(frames)})
        if   gt==1 and pred==1: TP+=1; lat.append(sm.trigger_frame if sm.trigger_frame is not None else len(frames))
        elif gt==0 and pred==1: FP+=1
        elif gt==1 and pred==0: FN+=1
        else: TN+=1
        print(f"  {name}: gt={gt} pred={pred} trig={sm.trigger_frame}")

    prec=TP/(TP+FP) if (TP+FP) else 0.0
    rec =TP/(TP+FN) if (TP+FN) else 0.0
    f1  =2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    acc =(TP+TN)/max(1,(TP+FP+FN+TN))
    EVAL={"dataset":"UR Fall Detection (URFD), cam0 RGB",
          "model":"MoveNet MultiPose Lightning (TF-Hub)",
          "state_machine":"faithful port of worksite_multi.html analyzePerson(), deployed defaults (sens=1.0)",
          "n_fall":N_FALL,"n_adl":N_ADL,"frame_stride":FRAME_STRIDE,
          "counts":{"TP":TP,"FP":FP,"FN":FN,"TN":TN},
          "precision":round(prec,4),"recall":round(rec,4),"f1":round(f1,4),"accuracy":round(acc,4),
          "detection_latency_frames":{"mean":(float(np.mean(lat)) if lat else None),
                                       "median":(float(np.median(lat)) if lat else None)},
          "caveat":"URFD = clean single-subject lab footage, fixed camera. Bounds generalization; != SME shop-floor. Evaluates pose->state-machine logic, not multi-person association. Thresholds unchanged from deployment."}
    (RUN_DIR/"eval_metrics.json").write_text(json.dumps(EVAL, indent=2, ensure_ascii=False))
    with open(RUN_DIR/"eval_per_sequence.csv","w",newline="") as fh:
        w=csv.DictWriter(fh, fieldnames=["seq","gt","pred","trigger_frame","frames"]); w.writeheader(); w.writerows(rows)
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    cm=np.array([[TN,FP],[FN,TP]]); mx=max(1,cm.max())
    fig,ax=plt.subplots(figsize=(3.5,3.3))  # publication B/W style (matches paper Fig.)
    for (r_,c_),v in np.ndenumerate(cm):
        sh=0.93-0.30*(v/mx)
        ax.add_patch(Rectangle((c_,1-r_),1,1,fc=(sh,sh,sh),ec="#1a1a1a",lw=1.2))
        ax.text(c_+0.5,1-r_+0.5,int(v),ha="center",va="center",fontsize=16,fontweight="bold",color="#000")
    ax.set_xlim(0,2); ax.set_ylim(0,2); ax.set_aspect("equal")
    ax.set_xticks([0.5,1.5]); ax.set_xticklabels(["pred ADL","pred Fall"],fontsize=9)
    ax.set_yticks([1.5,0.5]); ax.set_yticklabels(["true ADL","true Fall"],fontsize=9)
    ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title("URFD fall detection (%d fall + %d ADL)\nPrecision %.2f  Recall %.2f  F1 %.2f"%(N_FALL,N_ADL,prec,rec,f1),fontsize=9.5,pad=8)
    fig.savefig(RUN_DIR/"eval_confusion_matrix.png", dpi=300, bbox_inches="tight", pad_inches=0.08); plt.close(fig)
    print("\nEVAL:", json.dumps(EVAL, ensure_ascii=False, indent=2))
except Exception as e:
    import traceback; traceback.print_exc()
    print("\n[eval skipped]", repr(e))
    print("-> Harness/manifest still proceed. Re-run in Colab with TF + internet to produce the real fall-detection result.")
    EVAL = None


Loading MoveNet MultiPose Lightning (matches the deployed model) ...
  fall-01: gt=1 pred=1 trig=150
  fall-02: gt=1 pred=1 trig=81
  fall-03: gt=1 pred=1 trig=202
  fall-04: gt=1 pred=1 trig=41
  fall-05: gt=1 pred=1 trig=118
  fall-06: gt=1 pred=1 trig=57
  fall-07: gt=1 pred=1 trig=122
  fall-08: gt=1 pred=1 trig=51
  fall-09: gt=1 pred=1 trig=176
  fall-10: gt=1 pred=1 trig=63
  fall-11: gt=1 pred=1 trig=84
  fall-12: gt=1 pred=1 trig=54
  fall-13: gt=1 pred=0 trig=None
  fall-14: gt=1 pred=1 trig=39
  fall-15: gt=1 pred=1 trig=63
  fall-16: gt=1 pred=1 trig=43
  fall-17: gt=1 pred=0 trig=None
  fall-18: gt=1 pred=1 trig=42
  fall-19: gt=1 pred=1 trig=83
  fall-20: gt=1 pred=1 trig=68
  fall-21: gt=1 pred=0 trig=None
  fall-22: gt=1 pred=1 trig=40
  fall-23: gt=1 pred=0 trig=None
  fall-24: gt=1 pred=1 trig=50
  fall-25: gt=1 pred=0 trig=None
  fall-26: gt=1 pred=1 trig=36
  fall-27: gt=1 pred=0 trig=None
  fall-28: gt=1 pred=1 trig=50
  fall-29: gt=1 pred=0 trig=None
  fall-30: gt

## A-3 · 임계값 민감도 스윕 + A-4 · 구성요소 어블레이션 (추가 셀)
위의 본 평가(셀 4)가 끝난 뒤 실행하세요. **MoveNet 키포인트를 시퀀스별로 1회 캐시**한 뒤(추가 비용 = 본 평가 1회분, 이미 캐시돼 있으면 즉시), 상태기계만 고속 재생하여:
- **민감도 곡선**: F1 vs tilt(40–64°, 창 700ms 고정) / F1 vs 확인 창(300–1100ms, tilt 52° 고정)
- **운영점 산포도**: 35개 (tilt, 창) 조합의 TPR–FPR, 배포 운영점 강조
- **어블레이션**(동일 데이터·동일 조건): deployed / tilt-only / aspect-only / no-window

산출물: `sweep_results.csv` · `ablation_results.csv` · `fig_sweep_sensitivity.png` · `fig_sweep_operating.png` (RUN_DIR에 저장)
→ 논문 §IV에 "Threshold Sensitivity and Component Ablation" 소절 + 그림 1–2개로 반영.


In [13]:
# === A-3/A-4 · keypoint cache -> fast state-machine sweep (run AFTER cell 4) ===
import numpy as np, math, csv, json
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

def cache_keypoints(name):
    fz = CACHE / f"kps_{name}.npz"
    if fz.exists(): return fz
    frames = load_frames(name)
    arrs=[]; Wv=Hv=None
    for fr in frames:
        r = movenet_kps(fr)
        if r is None:
            arrs.append(np.zeros((17,3))); Wv = Wv or fr.shape[1]; Hv = Hv or fr.shape[0]
        else:
            arrs.append(r[0]); Wv, Hv = r[1], r[2]
    np.savez_compressed(fz, kps=np.array(arrs), W=Wv or 640, H=Hv or 480)
    return fz

class ParamFallSM:
    """FallSM(셀 4)와 동일 로직 — 임계값·사용 조건만 파라미터화."""
    def __init__(self, fps=30.0, tilt_thr=52.0, asp_thr=1.0, window_ms=700.0,
                 use_tilt=True, use_aspect=True):
        self.fps=fps; self.tilt_thr=tilt_thr; self.asp_thr=asp_thr
        self.window=window_ms; self.use_tilt=use_tilt; self.use_aspect=use_aspect
        self.t=0.0; self.downSince=None; self.hipHist=[]; self.lastKp=None
        self.triggered=False; self.trigger_frame=None; self.frame=-1
    def _v(self,kps,i):
        x,y,s=kps[i]; return (x,y) if s>0.3 else None
    def step(self,kps,W,H):
        self.frame+=1; self.t+=1.0/self.fps; now=self.t*1000.0
        shL=self._v(kps,5); shR=self._v(kps,6); hipL=self._v(kps,11); hipR=self._v(kps,12)
        if shL is None or shR is None: self.lastKp=kps; return
        shX=(shL[0]+shR[0])/2; shY=(shL[1]+shR[1])/2
        if hipL and hipR: hpX=(hipL[0]+hipR[0])/2; hpY=(hipL[1]+hipR[1])/2
        else:
            sw=math.hypot(shR[0]-shL[0],shR[1]-shL[1]); hpX=shX; hpY=shY+sw*1.4
        tilt=math.degrees(math.atan2(abs(shX-hpX),abs(shY-hpY)))
        xs=[k[0] for k in kps if k[2]>0.3]; ys=[k[1] for k in kps if k[2]>0.3]
        aspect=((max(xs)-min(xs))/max(1.0,(max(ys)-min(ys)))) if len(xs)>=2 else 0.0
        self.lastKp=kps
        isDown=(self.use_tilt and tilt>self.tilt_thr) or (self.use_aspect and aspect>self.asp_thr)
        if isDown:
            if not self.downSince: self.downSince=now
            if now-self.downSince>self.window and not self.triggered:
                self.triggered=True; self.trigger_frame=self.frame
        else: self.downSince=None

def replay(name, tilt_thr, asp_thr, window_ms, use_tilt=True, use_aspect=True):
    d=np.load(CACHE/f"kps_{name}.npz"); K=d["kps"]; Wv=float(d["W"]); Hv=float(d["H"])
    sm=ParamFallSM(fps=30.0/FRAME_STRIDE, tilt_thr=tilt_thr, asp_thr=asp_thr,
                   window_ms=window_ms, use_tilt=use_tilt, use_aspect=use_aspect)
    for kp in K: sm.step(kp,Wv,Hv)
    return 1 if sm.triggered else 0

# ---- 1) 캐시 패스 (이미 있으면 건너뜀) ----
seqs=[(f"fall-{i:02d}",1) for i in range(1,N_FALL+1)]+[(f"adl-{i:02d}",0) for i in range(1,N_ADL+1)]
print("caching keypoints (skips if cached)...")
for name,_ in seqs:
    try: cache_keypoints(name); print(" ", name, "ok")
    except Exception as e: print("  skip", name, e)

def eval_grid(tilt,window,use_tilt=True,use_aspect=True,asp=1.0):
    TP=FP=FN=TN=0
    for name,gt in seqs:
        if not (CACHE/f"kps_{name}.npz").exists(): continue
        pred=replay(name,tilt,asp,window,use_tilt,use_aspect)
        if   gt==1 and pred==1: TP+=1
        elif gt==0 and pred==1: FP+=1
        elif gt==1 and pred==0: FN+=1
        else: TN+=1
    prec=TP/(TP+FP) if TP+FP else 0.0; rec=TP/(TP+FN) if TP+FN else 0.0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
    fpr=FP/(FP+TN) if FP+TN else 0.0
    return dict(TP=TP,FP=FP,FN=FN,TN=TN,precision=round(prec,4),recall=round(rec,4),
                f1=round(f1,4),fpr=round(fpr,4))

# ---- 2) 스윕 ----
TILTS=[40,44,48,52,56,60,64]; WINDOWS=[300,500,700,900,1100]
rows=[]
for t in TILTS:
    for w in WINDOWS:
        m=eval_grid(t,w); m.update(tilt=t,window=w); rows.append(m)
        print(f"tilt={t} window={w}: F1={m['f1']} (TP{m['TP']} FP{m['FP']} FN{m['FN']} TN{m['TN']})")
with open(RUN_DIR/"sweep_results.csv","w",newline="") as fh:
    wtr=csv.DictWriter(fh,fieldnames=list(rows[0].keys())); wtr.writeheader(); wtr.writerows(rows)

# ---- 3) 어블레이션 (배포 임계값 고정) ----
abl=[("deployed (tilt+aspect, 700 ms)", dict(tilt=52,window=700)),
     ("tilt-only",                      dict(tilt=52,window=700,use_aspect=False)),
     ("aspect-only",                    dict(tilt=52,window=700,use_tilt=False)),
     ("no confirmation window",         dict(tilt=52,window=0))]
arows=[]
for label,kw in abl:
    m=eval_grid(**kw); m["variant"]=label; arows.append(m)
    print(label, "->", {k:m[k] for k in ("precision","recall","f1")})
with open(RUN_DIR/"ablation_results.csv","w",newline="") as fh:
    wtr=csv.DictWriter(fh,fieldnames=list(arows[0].keys())); wtr.writeheader(); wtr.writerows(arows)

# ---- 4) 그림 ----
f1_t=[next(r for r in rows if r["tilt"]==t and r["window"]==700)["f1"] for t in TILTS]
f1_w=[next(r for r in rows if r["tilt"]==52 and r["window"]==w)["f1"] for w in WINDOWS]
fig,ax=plt.subplots(1,2,figsize=(7.0,2.6))
ax[0].plot(TILTS,f1_t,"o-",color="#0072B2",ms=5); ax[0].axvline(52,ls="--",lw=0.9,color="0.4")
ax[0].set_xlabel("Tilt threshold (deg)"); ax[0].set_ylabel("F1"); ax[0].set_title("(a) F1 vs tilt (window = 700 ms)",fontsize=9)
ax[1].plot(WINDOWS,f1_w,"o-",color="#D55E00",ms=5); ax[1].axvline(700,ls="--",lw=0.9,color="0.4")
ax[1].set_xlabel("Confirmation window (ms)"); ax[1].set_ylabel("F1"); ax[1].set_title("(b) F1 vs window (tilt = 52°)",fontsize=9)
for a in ax: a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
fig.tight_layout(); fig.savefig(RUN_DIR/"fig_sweep_sensitivity.png",dpi=300,bbox_inches="tight"); plt.close(fig)

fig,ax=plt.subplots(figsize=(3.6,3.4))
ax.scatter([r["fpr"] for r in rows],[r["recall"] for r in rows],s=22,c="#999999",label="grid (35 settings)")
dep=next(r for r in rows if r["tilt"]==52 and r["window"]==700)
ax.scatter([dep["fpr"]],[dep["recall"]],s=70,c="#D55E00",zorder=5,label="deployed (52°, 700 ms)")
ax.set_xlabel("False-positive rate"); ax.set_ylabel("Recall (TPR)")
ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.legend(frameon=False,fontsize=7.5,loc="lower right")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout(); fig.savefig(RUN_DIR/"fig_sweep_operating.png",dpi=300,bbox_inches="tight"); plt.close(fig)
print("saved:", RUN_DIR/"sweep_results.csv", RUN_DIR/"fig_sweep_sensitivity.png", RUN_DIR/"fig_sweep_operating.png")


caching keypoints (skips if cached)...
  fall-01 ok
  fall-02 ok
  fall-03 ok
  fall-04 ok
  fall-05 ok
  fall-06 ok
  fall-07 ok
  fall-08 ok
  fall-09 ok
  fall-10 ok
  fall-11 ok
  fall-12 ok
  fall-13 ok
  fall-14 ok
  fall-15 ok
  fall-16 ok
  fall-17 ok
  fall-18 ok
  fall-19 ok
  fall-20 ok
  fall-21 ok
  fall-22 ok
  fall-23 ok
  fall-24 ok
  fall-25 ok
  fall-26 ok
  fall-27 ok
  fall-28 ok
  fall-29 ok
  fall-30 ok
  adl-01 ok
  adl-02 ok
  adl-03 ok
  adl-04 ok
  adl-05 ok
  adl-06 ok
  adl-07 ok
  adl-08 ok
  adl-09 ok
  adl-10 ok
  adl-11 ok
  adl-12 ok
  adl-13 ok
  adl-14 ok
  adl-15 ok
  adl-16 ok
  adl-17 ok
  adl-18 ok
  adl-19 ok
  adl-20 ok
  adl-21 ok
  adl-22 ok
  adl-23 ok
  adl-24 ok
  adl-25 ok
  adl-26 ok
  adl-27 ok
  adl-28 ok
  adl-29 ok
  adl-30 ok
  adl-31 ok
  adl-32 ok
  adl-33 ok
  adl-34 ok
  adl-35 ok
  adl-36 ok
  adl-37 ok
  adl-38 ok
  adl-39 ok
  adl-40 ok
tilt=40 window=300: F1=0.625 (TP25 FP25 FN5 TN15)
tilt=40 window=500: F1=0.6761 (TP24 FP17 

## A-2 · 제2 데이터셋 평가 — **GMDCSA-24 (권장, 완전 자동)**
아래 A-2a 셀이 Zenodo에서 **GMDCSA-24**(81 낙상 + 79 ADL, 4명·3개 가정환경, 저해상 웹캠, **CC-BY 4.0**)를
자동 다운로드→정리→`DS2_DIR` 지정까지 수행하므로, **사용자가 준비할 것은 없습니다.**
그다음 A-2b 셀이 배포 로직 그대로(재튜닝 없음) 전 클립을 평가합니다.
- 인용: E. Alam, A. Sufian, P. Dutta, M. Leo, I. A. Hameed, "GMDCSA-24: A dataset for human fall detection in videos," *Data in Brief*, 2024.
- 다른 데이터셋(Le2i·MCFD)을 쓰려면 A-2a를 건너뛰고 `DS2_DIR/falls/`·`DS2_DIR/adl/` 규약으로 직접 배치 후 A-2b만 실행.
- **DS2 전용 모드**(URFD 재평가 생략, 선택): 셀 1의 `"30"`·`"40"`을 `"0"`·`"0"`으로 바꾸고 Run all —
  셀 4는 모델 로드만 하고 즉시 통과하며, 스윕 셀은 0으로 채워진 무해한 출력만 냅니다. 이 모드의 zip에는 DS2 결과만 유효합니다.


In [14]:
# === A-2a · GMDCSA-24 자동 다운로드·정리 (Zenodo, CC-BY 4.0) ===
# 81 falls + 79 ADL, 4명·3개 가정환경, 저해상 웹캠 — Alam et al., Data in Brief 2024.
# 이 셀이 다운로드(1.1GB)→압축해제→falls/adl 정리→DS2_DIR 지정까지 수행합니다.
import os, json as _json, zipfile, shutil, urllib.request
from pathlib import Path

GM_DIR = BASE_OUTPUT / "_gmdcsa24"
DS2 = GM_DIR / "sorted"
if (DS2/"falls").exists() and any((DS2/"falls").iterdir()):
    print("GMDCSA-24 이미 준비됨:", DS2)
else:
    GM_DIR.mkdir(parents=True, exist_ok=True)
    api = _json.load(urllib.request.urlopen("https://zenodo.org/api/records/12921216"))
    f0 = api["files"][0]
    url = f0["links"]["self"]
    if not url.endswith("/content"): url = url.rstrip("/") + "/content"
    zp = GM_DIR / "gmdcsa24.zip"
    if not zp.exists() or zp.stat().st_size < 1e9:
        print("downloading", f0["key"], f"({f0['size']/1e9:.2f} GB) ...")
        urllib.request.urlretrieve(url, zp)
    print("extracting ...")
    ex = GM_DIR / "raw"
    if not ex.exists():
        with zipfile.ZipFile(zp) as z: z.extractall(ex)
    # 경로에 'fall'/'adl' 키워드로 분류 (데이터셋은 폴더명으로 라벨링됨)
    (DS2/"falls").mkdir(parents=True, exist_ok=True)
    (DS2/"adl").mkdir(parents=True, exist_ok=True)
    nf = na = 0
    for p in ex.rglob("*"):
        if p.suffix.lower() not in (".mp4",".avi",".mov"): continue
        # 최상위 폴더(저장소명에 'Fall-Detection' 포함)를 제외한 하위 경로로 판정, ADL 우선
        rel = p.relative_to(ex).parts
        sub = "/".join(rel[1:]).lower() if len(rel) > 1 else p.name.lower()
        if "adl" in sub:
            tgt = DS2/"adl"/f"{na:03d}_{p.name}"; na += 1
        elif "fall" in sub:
            tgt = DS2/"falls"/f"{nf:03d}_{p.name}"; nf += 1
        else:
            continue
        if not tgt.exists(): shutil.copy(p, tgt)
    print(f"sorted: falls={nf} adl={na} (기대치 ≈ 81 / 79)")
    if not (60 <= nf <= 100 and 60 <= na <= 100):
        print("!! 분류 수가 기대 범위를 벗어남 — raw 폴더 구조를 확인하세요:", ex)
os.environ["DS2_DIR"] = str(DS2)
print("DS2_DIR =", os.environ["DS2_DIR"])


downloading ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.0.zip (1.11 GB) ...
extracting ...
sorted: falls=79 adl=81 (기대치 ≈ 81 / 79)
DS2_DIR = /content/output/_gmdcsa24/sorted


In [15]:
# === A-2 · second-dataset evaluation (same deployed logic, no re-tuning) ===
DS2_DIR = os.environ.get("DS2_DIR", "")   # 예: /content/drive/MyDrive/datasets/le2i
if not DS2_DIR or not os.path.isdir(DS2_DIR):
    print("DS2_DIR 미지정 또는 없음 — 데이터 배치 후 os.environ['DS2_DIR']=... 지정하고 재실행")
else:
    import glob, cv2, numpy as np, json
    def eval_video(path):
        cap=cv2.VideoCapture(path); fps=cap.get(cv2.CAP_PROP_FPS) or 25.0
        sm=FallSM(fps=fps)     # 셀 4의 배포 상태기계 그대로
        while True:
            ok,fr=cap.read()
            if not ok: break
            r=movenet_kps(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
            if r is None: sm.step([(0,0,0.0)]*17, fr.shape[1], fr.shape[0])
            else: sm.step(r[0], r[1], r[2])
        cap.release(); return 1 if sm.triggered else 0
    TP=FP=FN=TN=0; per=[]
    for gt,sub in ((1,"falls"),(0,"adl")):
        for p in sorted(glob.glob(os.path.join(DS2_DIR,sub,"*"))):
            if not p.lower().endswith((".mp4",".avi",".mov")): continue
            pred=eval_video(p); per.append({"file":os.path.basename(p),"gt":gt,"pred":pred})
            if   gt==1 and pred==1: TP+=1
            elif gt==0 and pred==1: FP+=1
            elif gt==1 and pred==0: FN+=1
            else: TN+=1
            print(os.path.basename(p), "gt",gt,"pred",pred)
    prec=TP/(TP+FP) if TP+FP else 0; rec=TP/(TP+FN) if TP+FN else 0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0
    out={"dataset_dir":DS2_DIR,"counts":{"TP":TP,"FP":FP,"FN":FN,"TN":TN},
         "precision":round(prec,4),"recall":round(rec,4),"f1":round(f1,4),
         "note":"deployed thresholds, no re-tuning (sens=1.0)"}
    (RUN_DIR/"eval_ds2_metrics.json").write_text(json.dumps(out,indent=2))
    print(json.dumps(out,indent=2))


000_15.mp4 gt 1 pred 1
001_16.mp4 gt 1 pred 1
002_05.mp4 gt 1 pred 1
003_06.mp4 gt 1 pred 1
004_14.mp4 gt 1 pred 1
005_08.mp4 gt 1 pred 1
006_11.mp4 gt 1 pred 1
007_10.mp4 gt 1 pred 1
008_01.mp4 gt 1 pred 1
009_09.mp4 gt 1 pred 1
010_17.mp4 gt 1 pred 1
011_07.mp4 gt 1 pred 1
012_12.mp4 gt 1 pred 1
013_02.mp4 gt 1 pred 1
014_03.mp4 gt 1 pred 1
015_04.mp4 gt 1 pred 1
016_13.mp4 gt 1 pred 1
017_24.mp4 gt 1 pred 1
018_15.mp4 gt 1 pred 1
019_18.mp4 gt 1 pred 1
020_16.mp4 gt 1 pred 1
021_22.mp4 gt 1 pred 1
022_05.mp4 gt 1 pred 1
023_21.mp4 gt 1 pred 1
024_23.mp4 gt 1 pred 1
025_06.mp4 gt 1 pred 1
026_14.mp4 gt 1 pred 1
027_08.mp4 gt 1 pred 1
028_11.mp4 gt 1 pred 1
029_10.mp4 gt 1 pred 1
030_01.mp4 gt 1 pred 1
031_19.mp4 gt 1 pred 1
032_09.mp4 gt 1 pred 1
033_17.mp4 gt 1 pred 1
034_07.mp4 gt 1 pred 1
035_25.mp4 gt 1 pred 1
036_12.mp4 gt 1 pred 1
037_02.mp4 gt 1 pred 1
038_20.mp4 gt 1 pred 1
039_03.mp4 gt 1 pred 1
040_04.mp4 gt 1 pred 1
041_13.mp4 gt 1 pred 1
042_15.mp4 gt 1 pred 1
043_16.mp4 

In [16]:
# === run_manifest.json — provenance for this run (env, versions, measured numbers, content hashes) ===
import importlib
def _ver(mod):
    try: return importlib.import_module(mod).__version__
    except Exception: return None
def _node_ver():
    try: return subprocess.run(["node","--version"],capture_output=True,text=True).stdout.strip()
    except Exception: return None
def _git():
    try:
        base = DRIVE_REPO or (REPO_LOCAL if REPO_LOCAL.exists() else None)
        if not base: return None
        return subprocess.run(["git","-C",str(base),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip() or None
    except Exception: return None
def _sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for ch in iter(lambda: f.read(65536), b""): h.update(ch)
    return h.hexdigest()

contents=[]
for root,_,files in os.walk(RUN_DIR):
    for fn in files:
        fp=Path(root)/fn
        contents.append({"path":str(fp.relative_to(RUN_DIR)),"bytes":fp.stat().st_size,"sha256":_sha256(fp)})

MEASUREMENT_SCOPE = ("Post-processing CPU cost only, on randomly-generated inputs (33 random landmarks; 80x60 random RGBA). "
  "Model inference is browser-side and EXCLUDED. Not end-to-end FPS. DESIGN-TARGET metrics "
  "(e.g., 97.8% accuracy, 340% ROI, 65% injury reduction, 10-second 119 dispatch) are NOT measured here. "
  "The URFD fall-detection block (eval_metrics.json), when present, is the one measured accuracy result.")

manifest={
  "run_id": RUN_ID,
  "timestamp": RUN_TIMES,
  "seed": SEED,
  "environment": {
    "in_colab": IN_COLAB,
    "python": sys.version.split()[0],
    "node": _node_ver(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "packages": {m:_ver(m) for m in ["numpy","matplotlib","PIL","tensorflow","tensorflow_hub","cv2"]},
  },
  "git_commit": _git(),
  "measurement_scope": MEASUREMENT_SCOPE,
  "latency_benchmark": (BENCH if "BENCH" in globals() else None),
  "fall_detection_eval": (EVAL if "EVAL" in globals() else None),
  "contents": sorted(contents, key=lambda x: x["path"]),
}
(RUN_DIR/"run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print("wrote", RUN_DIR/"run_manifest.json", "(%d files)" % len(contents))
print(json.dumps({k:manifest[k] for k in ("run_id","timestamp","git_commit")}, indent=2, ensure_ascii=False))


wrote /content/output/run_20260723_135028/run_manifest.json (8 files)
{
  "run_id": "run_20260723_135028",
  "timestamp": {
    "utc_iso": "2026-07-23T04:50:28.601741+00:00",
    "kst_iso": "2026-07-23T13:50:28.601741+09:00",
    "tz_offset": "+09:00",
    "epoch": 1784782228.601741
  },
  "git_commit": null
}


In [17]:
# === Mirror to Drive (if mounted), zip the run, and download (Colab) — triple-copy insurance ===
if DRIVE_REPO:
    try:
        dst = DRIVE_REPO / "output" / RUN_ID
        shutil.copytree(RUN_DIR, dst, dirs_exist_ok=True)
        print("mirrored to Drive:", dst)
    except Exception as e:
        print("Drive mirror skipped:", e)

zip_base = str(BASE_OUTPUT / RUN_ID)
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(RUN_DIR))
print("zipped:", zip_path)

if IN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("download skipped:", e)
print("DONE — results for this run are in:", RUN_DIR)


zipped: /content/output/run_20260723_135028.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE — results for this run are in: /content/output/run_20260723_135028


## 실행 후

`EVAL:` 블록의 수치(precision/recall/F1/accuracy, latency)가 최종 결과입니다.
zip 안의 `eval_metrics.json`·`eval_per_sequence.csv`·`eval_confusion_matrix.png`를
`코드/output/run_<id>/`에 보관하고, 원고 §4.1·Table 3·Figure 4를 이 수치로 갱신하세요.
(부분집합 8+8 결과 run_20260627_222419와 나란히 인용하면 정직한 확장 서사가 됩니다.)